# From research PDFs to paper embeddings

This notebook runs a reproducible representation pipeline for a thematic map of private-equity research. It converts a folder of PDFs into one normalized vector per paper, while preserving paper identities and the text behind each vector.

The implementation is in `src/batill/`. Notebook cells expose the stages and their outputs; they do not contain a second implementation. Run cells in order with the project environment selected. Place the corpus PDFs in `data/pdfs/`. The configuration below uses `configs/pipeline.json`. Subfolders are supported.

The stages are **discover → extract → inspect → prepare inputs → embed → validate**. Extraction uses native PDF text and layout detection, with OCR disabled. The embedding stage uses a pinned Qwen3-Embedding-4B revision. Model inference is disabled by default in this notebook so the earlier stages can be reviewed on a machine with limited memory.

## 1. Configuration and input discovery

Configuration paths are relative to the project root. Input PDFs and generated outputs live in separate folders. Content hashes identify papers independently of their filenames, and exact duplicate files are recorded and skipped during extraction.

`configs/pipeline.json` reads PDFs from `data/pdfs/` and writes generated results to `outputs/corpus/`. Previous development PDFs and outputs have been archived outside the project.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'batill').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the batill project folder.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from batill import pipeline
from batill.extraction import discover_pdfs, load_documents
from batill.embedding import RuntimeConfig, load_plans
from batill.storage import read_json

CONFIG_FILE = ROOT / 'configs' / 'pipeline.json'
config = pipeline.load_config(CONFIG_FILE)
print('Python:', sys.executable)
print('Input:', config['pdf_dir'])
print('Output:', config['output_dir'])
print('Model:', config['embedding']['model'])
records = discover_pdfs(config['pdf_dir'])
display(pd.DataFrame(records).drop(columns='path'))

Python: /Users/fdedek/miniforge3/envs/ml/bin/python
Input: /Users/fdedek/projects/batill/data/pdfs
Output: /Users/fdedek/projects/batill/outputs/corpus
Model: Qwen/Qwen3-Embedding-4B


,filename,pdf_sha256,duplicate
0,Abraham et al. - 2024 - ESG Disclosures in the...,b87701a573df443374b263f59e2706fb6b0fb7c9286c6c...,False
1,Accounting Finance - 2011 - De Maeseneire - ...,2d3b2ba746ed713cfe70a748672f6eda554db7a371c1bb...,False
2,Acharya et al. - 2013 - Corporate Governance a...,d52328858a91e1b966a1afcbec8e586e9326a702a95519...,False
3,"Acharya und Johnson - 2010 - More insiders, mo...",288b181ec747cc2e900ef707239969720d711dfcbd0af6...,False
4,Achleitner et al. - 2012 - Structure and Deter...,2a21b697e4f775b6651ffcbee7ea664921c6067a5c4232...,False
...,...,...,...
315,Villalonga und Amit - 2009 - How Are U.S. Fami...,c61a8c87b4e4c926fcd5205556d0a0db8f9f0514405858...,False
316,Winton - 2003 - Institutional Liquidity Needs ...,c602e64ec46fdce5a09ab2776cc8a157c47a860fea78cd...,False
317,Winton und Yerramilli - 2008 - Entrepreneurial...,d38d926865319c50469c6c3aa415c5dad1db3d8914d420...,False
318,Wright et al. - 2009 - The Impact of Private E...,a8a19a13cf7444062bee2a550cdf4ae071911ee2454688...,False


## 2. Extract native PDF text and document structure

Marker identifies headings, paragraphs, tables and other blocks in reading order. OCR and formula transcription are disabled. The pipeline retains the raw structured result for traceability and exports a readable view that excludes detected equations, headers, footers and references. Marked inline formulas are replaced by `[FORMULA]`; unmarked mathematical fragments can remain.

This route assumes PDFs have a usable text layer. Scanned or damaged pages need separate handling. A successful converter call does not establish that the extracted text is complete or correctly ordered.

Extraction is cached by PDF content, extractor version and options. Each paper's status and elapsed time are saved. The first uncached paper includes model initialization and any required downloads; cached timings measure reading, not extraction speed.

In [ ]:
report = pipeline.extract(config)
display(pd.DataFrame(report).reindex(columns=[
    'filename', 'status', 'elapsed_seconds', 'retained_characters', 'quality_flags', 'error']))
fresh = [r['elapsed_seconds'] for r in report if r['status'] == 'extracted']
print(f'Fresh extraction time: {sum(fresh):.1f}s across {len(fresh)} papers')
print('Reports and readable text:', config['output_dir'] / 'extraction')

## 3. Inspect the retained prose

Compare each readable output with its source PDF. Inspect title accuracy, section boundaries, tables, missing text, residual formula fragments and the end of the reference section. The table contains block locations for tracing a problem back to the PDF; coordinates are not included in model inputs.

Change `document_index` to inspect another paper. Full Markdown files are in `extraction/readable/`; the printed excerpt is deliberately limited. Extraction failures must be resolved before preparing embeddings.

In [ ]:
from batill.documents import embedding_blocks

documents = load_documents(config['output_dir'] / 'extraction')
document_index = 0
document = documents[document_index]
retained = embedding_blocks(document, math_policy='omit')
print('Title:', document['title'])
print('Title source:', document['title_source'])
print('PDF ID:', document['pdf_sha256'])
print('Source PDF:', config['pdf_dir'] / document['source_filename'])
display(pd.DataFrame(retained).reindex(columns=['page', 'type', 'section', 'text']))
print('\n\n'.join(b['text'] for b in retained)[:15000])

## 4. Prepare exact model inputs

The input combines a task instruction, the extracted title and the retained paper text. The tokenizer comes from the same immutable revision as the embedding model. Token counts include the title, instruction and special tokens.

A paper that fits within 32,768 tokens uses one input. Longer papers are split at section/block boundaries; an oversized block is divided with recorded character offsets. No retained content is silently truncated. Exact strings and source ranges are saved under `prepared/`.

Set `selected_ids` to the paper hashes you want to include after inspection. `None` selects every successfully extracted unique paper; this is a selection convention, not an assertion that quality has been reviewed. The manifest selects the current corpus explicitly, so old cached files are not added automatically.
This notebook explicitly sets `SKIP_EMPTY_DOCUMENTS = True`: papers with no retained text are excluded, listed below, and recorded in `prepared/preparation_report.json` and the prepared manifest. Their PDFs and extraction files remain available for recovery. Set it to `False` to stop instead. This check does not detect incomplete or poor-quality text in otherwise nonempty papers.


In [2]:
selected_ids = None  # Or a list of inspected pdf_sha256 values.
SKIP_EMPTY_DOCUMENTS = True  # Record and exclude empty extractions; inspect the table below.
plans = pipeline.prepare(config, selected_ids=selected_ids, skip_empty=SKIP_EMPTY_DOCUMENTS)
preparation_report = read_json(config['output_dir'] / 'prepared' / 'preparation_report.json')
print(f"Prepared {len(plans)} papers; excluded {len(preparation_report['excluded'])} empty papers.")
if preparation_report['excluded']:
    display(pd.DataFrame(preparation_report['excluded']))
display(pd.DataFrame([{
    'title': p['title'], 'pdf_sha256': p['pdf_sha256'],
    'segments': len(p['segments']),
    'total_input_tokens': sum(s['token_count'] for s in p['segments']),
    'largest_segment_tokens': max(s['token_count'] for s in p['segments']),
} for p in plans]))
print('Prepared text:', config['output_dir'] / 'prepared' / 'readable')

/Users/fdedek/projects/batill/src/batill/pipeline.py:62: UserWarning: Excluding 1 empty document(s):
- Kovenock und Phillips - 1997 - Capital Structure and Product Market Behavior An Examination of Plant Exit and Investment Decisions.pdf (574ef7922d26910a58a84672526557d66acc3797e672b1c3099d0bd6354fa22c)
Report: /Users/fdedek/projects/batill/outputs/corpus/prepared/preparation_report.json
  return prepare_corpus(documents, tokenizer, embedding, revision, output / 'prepared',


Prepared 318 papers; excluded 1 empty papers.


,pdf_sha256,source_filename,title,reason,quality_flags
0,574ef7922d26910a58a84672526557d66acc3797e672b1...,Kovenock und Phillips - 1997 - Capital Structu...,Kovenock und Phillips - 1997 - Capital Structu...,no_usable_blocks,"[no_usable_text, contains_detected_formulas]"


,title,pdf_sha256,segments,total_input_tokens,largest_segment_tokens
0,ESG Disclosures in the Private Equity Industry,b87701a573df443374b263f59e2706fb6b0fb7c9286c6c...,1,25123,25123
1,What drives leverage in leveraged buyouts? An ...,2d3b2ba746ed713cfe70a748672f6eda554db7a371c1bb...,1,17034,17034
2,Corporate Governance and Value Creation: Evide...,d52328858a91e1b966a1afcbec8e586e9326a702a95519...,1,30218,30218
3,"More insiders, more insider trading: Evidence ...",288b181ec747cc2e900ef707239969720d711dfcbd0af6...,31,34251,6092
4,Structure and Determinants of Financial Covena...,2a21b697e4f775b6651ffcbee7ea664921c6067a5c4232...,1,29103,29103
...,...,...,...,...,...
313,How Are U.S. Family Firms Controlled?,c61a8c87b4e4c926fcd5205556d0a0db8f9f0514405858...,1,30397,30397
314,Institutional Liquidity Needs and the Structur...,c602e64ec46fdce5a09ab2776cc8a157c47a860fea78cd...,1,25933,25933
315,Entrepreneurial finance: Banks versus venture ...,d38d926865319c50469c6c3aa415c5dad1db3d8914d420...,1,24803,24803
316,The Impact of Private Equity and Buyouts on Em...,a8a19a13cf7444062bee2a550cdf4ae071911ee2454688...,1,6574,6574


Prepared text: /Users/fdedek/projects/batill/outputs/corpus/prepared/readable


In [3]:
paper_index, segment_index = 0, 0
segment = plans[paper_index]['segments'][segment_index]
print(segment['text'][:12000])
print('Input tokens:', segment['token_count'])
display(pd.DataFrame(segment['sources']))

Instruct: Represent this research paper for clustering by its economic topic, research question, and mechanisms.
Query: Title: ESG Disclosures in the Private Equity Industry

ESG Disclosures in the Private Equity Industry

JEFFERSON ABRAHAM, ∗ MARCEL OLBERT , ∗ AND FLORIN VASVARI ∗

Received 1 November 2022; accepted 19 July 2024

ABSTRACT

This paper offers the first systematic evidence on environmental, social, and governance (ESG) disclosures provided by a large global sample of private

∗London Business School

Accepted by Valeri Nikolaev. The paper has greatly benefited from the thoughtful feedback from an anonymous reviewer and the associate editor. We acknowledge the generous data support provided by Preqin and thank the Financial Accounting and Reporting (FARS) Section of the American Accounting Association for the 2024 FARS Midyear Meeting Best Paper Award. We thank Brian Baik, Aymeric Bellon, Jannis Bischof, Thomas Bourveau, Patricia Breuer, Alper Darendeli, Svenja Dube (disc

,block_id,page,section,start,end
0,/page/0/SectionHeader/4,1,[ESG Disclosures in the Private Equity Industry],0,46
1,/page/0/Text/5,1,[ESG Disclosures in the Private Equity Industry],0,59
2,/page/0/Text/6,1,[ESG Disclosures in the Private Equity Industry],0,47
3,/page/0/SectionHeader/7,1,[ESG Disclosures in the Private Equity Industr...,0,8
4,/page/0/Text/8,1,[ESG Disclosures in the Private Equity Industr...,0,151
...,...,...,...,...,...
408,/page/45/Text/5,46,"[appendix a: variable definitions, appendix c:...",0,666
409,/page/45/Footnote/7,46,"[appendix a: variable definitions, appendix c:...",0,628
410,/page/46/Text/3,47,"[appendix a: variable definitions, appendix c:...",0,46
411,/page/46/Text/4,47,"[appendix a: variable definitions, appendix c:...",0,61


## 5. Generate embeddings on a sufficiently resourced machine

Set `RUN_EMBEDDING = True` only on the machine where inference should run. The configuration specifies the device, precision and attention backend explicitly. The code does not silently switch to CPU or shorten inputs when resources are insufficient. For the development Mac, leave this cell disabled.

The model encodes one segment at a time. For a paper with multiple segments, their normalized vectors are averaged using segment content-token counts as weights, and the paper vector is normalized again. This aggregation is a documented modelling choice, not evidence that all sections contribute equally useful thematic information.

Completed papers are cached independently. Reruns validate and reuse them. If all required vectors already exist, this stage can assemble the final matrix without loading model weights. Outputs use relative paths and record the actual execution environment, so the same files can be produced on another machine and copied here.

### Optional multi-GPU execution

Set `GPU_WORKERS = 4` to distribute complete papers across four CUDA GPUs on one machine. Each worker loads its own model and processes all segments of its assigned papers. The assignment balances estimated total input tokens; the final matrix keeps the original manifest order. Workers have isolated cache/report folders, and merging occurs only after every worker finishes successfully. This increases corpus throughput; it does not combine GPU memory to fit a single paper. Leave `GPU_WORKERS = 1` for the sequential path. Actual four-GPU performance has not yet been benchmarked.

In [ ]:
RUN_EMBEDDING = False  # Keep False on the development Mac.
GPU_WORKERS = 1  # Set to 4 on a machine with four visible CUDA GPUs.
if RUN_EMBEDDING:
    runtime = RuntimeConfig(**config['runtime'])
    destination = pipeline.embed(config['output_dir'], runtime, workers=GPU_WORKERS)
    print('Saved embeddings:', destination)
else:
    print('Inference skipped. Continue below once matching embeddings are present.')

## 6. Validate and load the resulting representation

Validation checks the prepared-input hashes, model settings, paper order, dimensions, finite values, normalized paper vectors and segment aggregation. These are integrity checks; they do not measure the scientific quality of the representation.

The final files always live at `embeddings/embeddings.npz` and `embeddings/manifest.json` inside the configured output folder. Keep the manifest with the matrix: row order and PDF hashes link vectors to papers. These are the inputs for the later clustering and map evaluation work.

In [4]:
import numpy as np

summary = pipeline.validate_outputs(config['output_dir'])
print(summary)
folder = config['output_dir'] / 'embeddings'
manifest = read_json(folder / 'manifest.json')
with np.load(folder / 'embeddings.npz', allow_pickle=False) as saved:
    vectors = saved['embeddings']
    paper_ids = saved['pdf_sha256']
print('Matrix:', vectors.shape)
print('Vector norms:', np.linalg.norm(vectors, axis=1))
display(pd.DataFrame(manifest['rows'])[['title', 'pdf_sha256', 'source_filename']])

{'papers': 318, 'dimensions': 2560, 'corpus_id': '16634a146c55a3cd810476d8586ec71e2084f65fa8839a9f418fe79d7ee64e70', 'matrix': '/Users/fdedek/projects/batill/outputs/corpus/embeddings/embeddings.npz'}
Matrix: (318, 2560)
Vector norms: [1.         1.         1.         1.         0.99999994 1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.         1.         0.99999994 1.
 1.         1.         1.         0.99999994 0.99999994 1.
 1.         1.         1.         1.         0.99999994 1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.         1.         1.         1.
 0.99999994 1.         1.         0.99999994 1.         1

,title,pdf_sha256,source_filename
0,ESG Disclosures in the Private Equity Industry,b87701a573df443374b263f59e2706fb6b0fb7c9286c6c...,Abraham et al. - 2024 - ESG Disclosures in the...
1,What drives leverage in leveraged buyouts? An ...,2d3b2ba746ed713cfe70a748672f6eda554db7a371c1bb...,Accounting Finance - 2011 - De Maeseneire - ...
2,Corporate Governance and Value Creation: Evide...,d52328858a91e1b966a1afcbec8e586e9326a702a95519...,Acharya et al. - 2013 - Corporate Governance a...
3,"More insiders, more insider trading: Evidence ...",288b181ec747cc2e900ef707239969720d711dfcbd0af6...,"Acharya und Johnson - 2010 - More insiders, mo..."
4,Structure and Determinants of Financial Covena...,2a21b697e4f775b6651ffcbee7ea664921c6067a5c4232...,Achleitner et al. - 2012 - Structure and Deter...
...,...,...,...
313,How Are U.S. Family Firms Controlled?,c61a8c87b4e4c926fcd5205556d0a0db8f9f0514405858...,Villalonga und Amit - 2009 - How Are U.S. Fami...
314,Institutional Liquidity Needs and the Structur...,c602e64ec46fdce5a09ab2776cc8a157c47a860fea78cd...,Winton - 2003 - Institutional Liquidity Needs ...
315,Entrepreneurial finance: Banks versus venture ...,d38d926865319c50469c6c3aa415c5dad1db3d8914d420...,Winton und Yerramilli - 2008 - Entrepreneurial...
316,The Impact of Private Equity and Buyouts on Em...,a8a19a13cf7444062bee2a550cdf4ae071911ee2454688...,Wright et al. - 2009 - The Impact of Private E...


## Interpretation and next steps

Successful extraction and vector validation establish a functional pipeline. Before fitting a final map, inspect nearest neighbours, select interpretable paper pairs and assess whether distances reflect the economic themes and mechanisms relevant to the thesis. The earlier nine-paper test validated the software path; the full corpus still requires scientific evaluation.

`Clustering PDFs.ipynb` loads validated paper-level vectors from `outputs/corpus_sections/` by default. Set `OUTPUT_DIR` to `ROOT / 'outputs' / 'corpus'` to inspect the full-paper vectors from this pipeline. After the embeddings are available, run its setup and loading cells, then the hierarchical clustering section to inspect the dendrogram and ordered cosine-similarity heatmap. Its analysis cells are left unexecuted for you to run.